# TWO WORLDS — Kaggle GPU Visual Studio (first real generation)

**Goal:** free-quota test of high-quality illustration with FLUX.2 klein 4B, plus English voice sample via Kokoro-82M, without writing any screenplay.

**Important:** GPU and internet must be enabled in Kaggle Settings. Free GPU availability and VRAM are not guaranteed. Kaggle T4/P100 lack bfloat16; the code uses float16 and CPU offloading. This can still exceed GPU/RAM/disk quotas. Run only small tests first, and stop the session afterward.

Every image/voice remains **unreviewed**. This notebook does NOT merge a PR, post to TikTok, or mark a finished episode approved.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path("/kaggle/working/Two-Worlds-Studio")
ART_DIR = Path("/kaggle/working/two_worlds_art")
VOICE_DIR = Path("/kaggle/working/two_worlds_voice")
ENABLE_GPU_IMAGE = False  # IMPORTANT: set True only after enabling GPU and accepting model download
ENABLE_KOKORO_VOICE = False  # set True to synthesize a short English voice sample
BODY = "earth_mira"
VIEW = "front"
REFERENCE = None  # optionally: "/kaggle/input/your-reference/approved_mira.png" (owner-approved only)
print("Python:",sys.version.split()[0])
print("GPU inference opt-in:", ENABLE_GPU_IMAGE, "| voice:", ENABLE_KOKORO_VOICE)
try:
    import torch
    print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(none)")
except ImportError:
    print("torch unavailable")

## 1. Download the actual TWO WORLDS source branch

The GitHub repository must be accessible. We clone **only the separate visual production branch**, never Williams projects. The branch can be merged only after review.


In [ ]:
if not ROOT.is_dir():
    subprocess.run(["git","clone","--depth","1","--branch","feat/kaggle-free-gpu-and-kokoro",
                    "https://github.com/19wolfalone92/Two-Worlds-Studio.git",str(ROOT)],check=True)
else:
    print("Existing local checkout found:", ROOT)
os.chdir(ROOT)
print("Repository:", ROOT)

## 2. Offline safety check: plans only, no model download

This checks the storyboard art request and optional voice plan. The preview E01 is a **technical fixture**, not a Work-approved screenplay.


In [ ]:
subprocess.run([sys.executable,"-m","studio.kaggle_flux","--mode","character",
                "--body",BODY,"--view",VIEW,"--dry-run",
                "--output-dir",str(ART_DIR)],check=True)
subprocess.run([sys.executable,"-m","studio.kokoro_tts","--text",
                "This is a technical sound test for the Two Worlds studio.",
                "--speaker","Mira","--dry-run","--output-dir",str(VOICE_DIR)],check=True)

## 3. FLUX.2 klein 4B: first real reference portrait (optional)

- Free GPU session must be running.
- Download requires Internet and free disk space; model size is large.
- The code checks CUDA, uses float16 on T4/P100 and enables CPU offload.
- It generates **one** image. If VRAM fails, use smaller dimensions (e.g. 512×768).
- A seed and prompt are NOT proof of same face. Approve front portrait first, then use it as an editing reference for other angles.


In [ ]:
if ENABLE_GPU_IMAGE:
    subprocess.run([sys.executable,"-m","pip","install","-q",
                    "git+https://github.com/huggingface/diffusers.git",
                    "transformers","accelerate","safetensors","sentencepiece"],check=True)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Activate GPU in Kaggle Settings > Accelerator and restart notebook.")
    command=[sys.executable,"-m","studio.kaggle_flux","--mode","character",
             "--body",BODY,"--view",VIEW,"--output-dir",str(ART_DIR),
             "--width","576","--height","1024","--steps","4"]
    if REFERENCE:
        command+=["--reference",str(REFERENCE)]
    subprocess.run(command,check=True)
else:
    print("GPU model download skipped. Change ENABLE_GPU_IMAGE=True to run actual inference.")

## 4. Stable character turnarounds (optional, requires an approved front reference)

After visually approving the front image, use it as `REFERENCE` and set `VIEW='three_quarter'` or `VIEW='profile'`. Rerun only the generation cell. Do NOT pretend likeness was measured automatically.


## 5. Kokoro-82M: real English test voice (optional)

Runs the original Kokoro-82M preset voice `af_heart`. These are voice prototypes, **not** selected final cast approvals. Model card: Apache-2.0.

You may run this even without GPU on sufficiently capable CPU, but dependency/model download and installation can take time.


In [ ]:
if ENABLE_KOKORO_VOICE:
    subprocess.run([sys.executable,"-m","pip","install","-q","kokoro>=0.9.2","soundfile"],check=True)
    subprocess.run(["bash","-lc","command -v espeak-ng >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq espeak-ng)"],check=True)
    subprocess.run([sys.executable,"-m","studio.kokoro_tts","--text",
                    "This is a technical sound test for the Two Worlds studio.",
                    "--speaker","Mira","--output-dir",str(VOICE_DIR)],check=True)
else:
    print("Voice generation skipped. Change ENABLE_KOKORO_VOICE=True to enable.")

## 6. Package only the actual created files

The zip is available in Kaggle's **Output** panel and is downloaded manually. Do not publish it from an unattended account. The repository stores small metadata, not large raw model weights or unpublished private image references.


In [ ]:
import zipfile
zip_path=Path("/kaggle/working/TWO_WORLDS_KAGGLE_OUTPUTS.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for directory in (ART_DIR,VOICE_DIR):
        if directory.exists():
            for path in directory.rglob("*"):
                if path.is_file():
                    z.write(path,arcname=str(path.relative_to(Path("/kaggle/working"))))
print("Package:",zip_path,"size:",zip_path.stat().st_size,"bytes")
print("No output is automatically published or uploaded to GitHub.")

## Pass / fail conditions

- PASS bootstrap: repository clone and dry-run succeed without GPU or model downloads.
- PASS FLUX: one real PNG plus sidecar JSON exist, reported GPU name, model ID and valid SHA-256; a person approves artistic quality.
- PASS consistent character: approved front portrait can condition additional three-quarter/profile images, and an artist confirms identity similarity. Not assumed from prompt text.
- PASS Kokoro: generated WAV has playable English speech and timing works; voices do not automatically become approved.
- FAIL: no actual GPU session/model download/approved artwork — the studio cannot honestly claim final professional readiness.

When ready, deliver the **Work-approved script and licensed art layers** to the repository's `studio.asset_ingest` and run the existing compositor. Future scheduled releases still require human review.
